In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIGURATION PK/PD (Modèle de Schnider) ---
V1, V2, V3 = 4.27, 18.9, 238.0
k10, k12, k21, k13, k31 = 0.38, 0.30, 0.20, 0.19, 0.0035
ke0 = 0.17  # Constante de transfert vers le site d'effet [8]

# Paramètres PD (Equation de Hill pour le BIS)
BIS_0, BIS_MAX, EC50, HILL = 95.0, 75.0, 3.5, 2.5 # [9]
BIS_TARGET = 50.0

# --- CONFIGURATION RL ---
ACTIONS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0] # Version réduite pour le calcul [10]
GAMMA = 0.69 # Facteur de remise [11]
BINS_PER_FEAT = 10
# 2 features (BIS error and deltaBIS), each with BINS_PER_FEAT bins
NUM_STATES = BINS_PER_FEAT**2  # 100 states

# --- ARTIFACT PERSISTENCE ---
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DP_PATH = ARTIFACTS_DIR / "dp_bis_deltabis_agent.npz"

def get_fuzzy_features(bis_error, deltabis):
    """Fuzzification using 2 membership functions: negative and positive

    Args:
        bis_error: BIS - TARGET (can be negative or positive)
        deltabis: BIS(t+1) - BIS(t-1)

    Returns:
        [is_negative, is_positive] for each feature
    """
    # For BIS error: scale by 20 (typical error range ±40)
    error_scaled = np.clip(bis_error / 20.0, -1, 1)
    mu_error_neg = max(0, -error_scaled)  # Negative membership
    mu_error_pos = max(0, error_scaled)   # Positive membership

    # For deltaBIS: scale by 20 (typical range ±40)
    delta_scaled = np.clip(deltabis / 20.0, -1, 1)
    mu_delta_neg = max(0, -delta_scaled)  # Negative membership
    mu_delta_pos = max(0, delta_scaled)   # Positive membership

    # Return 2 features: [error_membership, delta_membership]
    # Each can be considered as negative or positive
    return np.array([mu_error_neg, mu_error_pos, mu_delta_neg, mu_delta_pos])

def state_to_idx(features):
    """Map the 2D fuzzy feature vector to an integer index in 0..NUM_STATES-1.

    Use 2 main features: error and deltaBIS
    Each discretized to BINS_PER_FEAT bins
    """
    features = np.asarray(features)
    if features.shape != (4,):
        raise ValueError(f"features must be length-4 vector, got shape {features.shape}")

    # Use error negative/positive and delta negative/positive
    # Combine them: error dim (0 or 1) and delta dim (0 or 1)
    # This gives us 2x2 = 4 main states
    # But we want to use 10x10 = 100 states with fuzzy bins

    # Discretize each of the 2 main features (error and delta)
    error_feat = (features[1] - features[0]) * (BINS_PER_FEAT - 1)  # neg to pos
    delta_feat = (features[3] - features[2]) * (BINS_PER_FEAT - 1)  # neg to pos

    error_bin = int(np.clip(error_feat, 0, BINS_PER_FEAT - 1))
    delta_bin = int(np.clip(delta_feat, 0, BINS_PER_FEAT - 1))

    idx = error_bin * BINS_PER_FEAT + delta_bin
    return idx

def get_ce_from_error(bis_error):
    """Inverse de la fonction PD pour estimer Ce à partir du BIS error"""
    bis = BIS_ERROR + BIS_TARGET
    # Inversion de BIS = BIS_0 - BIS_MAX * (Ce^H / (Ce^H + EC50^H))
    ratio = (BIS_0 - bis) / BIS_MAX
    ratio = np.clip(ratio, 0.01, 0.99)
    ce = EC50 * (ratio / (1 - ratio))**(1/HILL)
    return ce

# --- DP ALGORITHM ---
# Always compute/update agent
loaded_V = None
if DP_PATH.exists():
    try:
        data = np.load(DP_PATH)
        loaded_V = data["V"]
        print(f"Found existing agent at {DP_PATH}, will warm-start Value Iteration.")
    except Exception as e:
        print(f"Could not load existing agent: {e}")

print("Computing Transition Matrix (P) and Rewards (R)...")
P = np.zeros((NUM_STATES, len(ACTIONS)), dtype=int)
R = np.zeros((NUM_STATES, len(ACTIONS)))

# Iterate directly over states and actions
for s in range(NUM_STATES):
    # Decode state index to bins
    error_bin = s // BINS_PER_FEAT
    delta_bin = s % BINS_PER_FEAT

    # Convert bins to feature values
    error_feat = (error_bin / (BINS_PER_FEAT - 1)) if BINS_PER_FEAT > 1 else 0.5
    delta_feat = (delta_bin / (BINS_PER_FEAT - 1)) if BINS_PER_FEAT > 1 else 0.5

    # Map back to actual error and deltaBIS values (approximately)
    bis_error = (error_feat * 2 - 1) * 30  # Range approximately -30 to 30
    deltabis = (delta_feat * 2 - 1) * 30   # Range approximately -30 to 30

    for a_idx, infusion in enumerate(ACTIONS):
        # Simulate next state based on current state and action
        # Higher infusion -> BIS decreases -> error becomes more positive
        bis_error_next = bis_error - (infusion / max(ACTIONS)) * 15

        # deltaBIS change based on infusion and current error
        # Infusion affects BIS change rate
        deltabis_next = deltabis - (infusion / max(ACTIONS)) * 10

        # Clip to valid ranges
        bis_error_next = np.clip(bis_error_next, -30, 30)
        deltabis_next = np.clip(deltabis_next, -30, 30)

        # Get next state
        features_next = get_fuzzy_features(bis_error_next, deltabis_next)
        s_next = state_to_idx(features_next)

        P[s, a_idx] = s_next
        # Reward: penalize large errors and large deltaBIS
        R[s, a_idx] = -(abs(bis_error_next) + 0.5 * abs(deltabis_next))

# 2. ALGORITHME DE PROGRAMMATION DYNAMIQUE (Value Iteration)
print("Exécution de Value Iteration...")
if loaded_V is not None and loaded_V.shape == (NUM_STATES,):
    V = loaded_V.copy()
else:
    V = np.zeros(NUM_STATES)

for i in range(10000):
    V_old = V.copy()
    # Vectorized computation: Q(s,a) = R(s,a) + gamma * V(P(s,a))
    Q_vals = R + GAMMA * V[P]  # Shape: (NUM_STATES, len(ACTIONS))
    V = np.max(Q_vals, axis=1)  # Max over actions

    if np.max(np.abs(V - V_old)) < 1e-4:
        print(f"Converged at iteration {i}")
        break

    if (i + 1) % 100 == 0:
        print(f"  Iteration {i+1}, max change: {np.max(np.abs(V - V_old)):.6f}")

# 3. EXTRACTION DE LA POLITIQUE OPTIMALE
print("Extracting optimal policy...")
Q_vals = R + GAMMA * V[P]
policy = np.argmax(Q_vals, axis=1).astype(int)

# --- SAVE DP AGENT ---
np.savez_compressed(
    DP_PATH,
    V=V,
    policy=policy,
    P=P,
    R=R,
    actions=np.array(ACTIONS, dtype=float),
    gamma=np.array([GAMMA], dtype=float),
)
print(f"Saved DP agent to {DP_PATH}")

# --- VISUALISATION ---
NUM_EVAL_EPISODES = 10
EP_LEN = 720            # episode length in time steps (12 minutes = 720 steps of 5s each)
DT = 5 / 60             # time step in minutes (5s)
np.random.seed(0)

# Plot policy as function of BIS error and deltaBIS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1D slices: Policy vs BIS error (with fixed deltaBIS)
error_range = np.linspace(-30, 30, 50)
deltabis_fixed = 0.0

chosen_doses_error = []
for err in error_range:
    features = get_fuzzy_features(err, deltabis_fixed)
    s = state_to_idx(features)
    chosen_doses_error.append(ACTIONS[policy[s]])

axes[0].plot(error_range, chosen_doses_error, color='blue', lw=2)
axes[0].set_title("Policy vs BIS Error (deltaBIS=0)")
axes[0].set_xlabel("BIS Error (BIS - Target)")
axes[0].set_ylabel("Infusion Rate (ml/min)")
axes[0].grid(True, linestyle='--')
axes[0].axvline(0, color='red', linestyle='-', alpha=0.3, label="Target")

# 1D slices: Policy vs deltaBIS (with fixed BIS error)
deltabis_range = np.linspace(-30, 30, 50)
error_fixed = 0.0

chosen_doses_delta = []
for db in deltabis_range:
    features = get_fuzzy_features(error_fixed, db)
    s = state_to_idx(features)
    chosen_doses_delta.append(ACTIONS[policy[s]])

axes[1].plot(deltabis_range, chosen_doses_delta, color='green', lw=2)
axes[1].set_title("Policy vs deltaBIS (Error=0)")
axes[1].set_xlabel("deltaBIS (BIS(t+1) - BIS(t-1))")
axes[1].set_ylabel("Infusion Rate (ml/min)")
axes[1].grid(True, linestyle='--')
axes[1].axvline(0, color='red', linestyle='-', alpha=0.3, label="Target")

plt.tight_layout()
plt.show()

# Evaluate policy with several simulated episodes
print(f"Evaluating policy for {NUM_EVAL_EPISODES} episodes, each {EP_LEN} steps...")
all_bis = np.zeros((NUM_EVAL_EPISODES, EP_LEN))
all_actions = np.zeros((NUM_EVAL_EPISODES, EP_LEN))
c0, c1, c2 = 0.0, 0.0, 0.0

for ep in range(NUM_EVAL_EPISODES):
    bis_prev = BIS_TARGET if ep > 0 else (BIS_TARGET - 5.0)
    bis_curr = bis_prev
    bis_prev_prev = bis_prev
    ce = 0.0  # start at baseline

    for t in range(EP_LEN):
        bis_error = bis_curr - BIS_TARGET
        deltabis = bis_curr - bis_prev_prev

        features = get_fuzzy_features(bis_error, deltabis)
        s = state_to_idx(features)
        a_idx = int(policy[s])
        infusion = ACTIONS[a_idx]

        # 3-compartment PK Euler update
        dt = DT
        dc0 = (infusion / V1) - (k10 + k12 + k13) * c0 + k21 * c1 + k31 * c2
        dc1 = k12 * c0 - k21 * c1
        dc2 = k13 * c0 - k31 * c2

        c0 += dc0 * dt
        c1 += dc1 * dt
        c2 += dc2 * dt

        ce += ke0 * (c0 - ce) * dt

        # PD: BIS from effect-site
        bis = BIS_0 - BIS_MAX * (ce**HILL / (ce**HILL + EC50**HILL))

        # Update history
        bis_prev_prev = bis_prev
        bis_prev = bis_curr
        bis_curr = bis

        all_bis[ep, t] = bis
        all_actions[ep, t] = infusion

# Plot BIS trajectories
mean_bis = all_bis.mean(axis=0)
std_bis = all_bis.std(axis=0)

t = np.arange(EP_LEN) * (DT * 60)  # time in seconds
plt.figure(figsize=(10, 5))
plt.plot(t, mean_bis, color='blue', label='Mean BIS')
plt.fill_between(t, mean_bis - std_bis, mean_bis + std_bis, color='blue', alpha=0.2, label='±1 std')

for ep in range(min(3, NUM_EVAL_EPISODES)):
    plt.plot(t, all_bis[ep], alpha=0.6, lw=1, label=f'Episode {ep+1}')

plt.axhline(BIS_TARGET, color='red', linestyle='--', label='Target BIS')
plt.xlabel('Time (s)')
plt.ylabel('BIS')
plt.title('Policy Evaluation: BIS trajectories (mean ± std and samples)')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

# Plot infusion traces
plt.figure(figsize=(10, 4))
for ep in range(min(4, NUM_EVAL_EPISODES)):
    plt.step(t, all_actions[ep], where='post', alpha=0.8, label=f'Ep {ep+1}')
plt.xlabel('Time (s)')
plt.ylabel('Infusion (ml/min)')
plt.title('Actions taken by policy during evaluation episodes')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()


# EVALUATION ON POPULATION DATASET

In [ ]:
import pandas as pd
import random

# 1. Data Loading and Preprocessing
def load_data(path: str):
    return pd.read_csv(path)

def preprocess_data(df: pd.DataFrame):
    df = df.copy()
    def parse_age(x):
        try:
            parts = str(x).strip().split(" ")
            if len(parts) < 2: return 50
            low = int(parts[1])
            if len(parts) > 3:
                if parts[3] == 'older':
                    high = 100
                else:
                    high = int(parts[3])
                return int(random.randrange(low, high))
            return int(low)
        except Exception:
            return 50

    if "AgeCategory" in df.columns:
        df.loc[:,"AgeCategory"] = df["AgeCategory"].apply(parse_age)
    return df

def schnider_model(age: int, weight: float, height: float, sex: str) -> dict:
    sex = str(sex).lower()
    if sex == "male":
        lbm = 1.10 * weight - 128 * (weight ** 2) / (height ** 2)
    else:
        lbm = 1.07 * weight - 148 * (weight ** 2) / (height ** 2)

    V1 = 4.27
    V2 = 18.9 - 0.391 * (age - 53)
    V3 = 238.0

    k10 = 0.443 + 0.0107 * (weight - 77) - 0.0159 * (lbm - 59) + 0.0062 * (height - 177)
    k12 = 0.302 - 0.0056 * (age - 53)
    k13 = 0.196
    k21 = (1.29 - 0.024 * (age - 53)) / V2
    k31 = 0.0035
    ke0 = 0.456

    A = np.array([
        [-(k10 + k12 + k13),  k21,  k31,   0   ],
        [        k12,         -k21,   0,    0   ],
        [        k13,          0,   -k31,   0   ],
        [        ke0,          0,    0,   -ke0  ]
    ], dtype=float)
    B = np.array([[1 / V1], [0], [0], [0]])
    return {"A": A, "B": B}

def generate_schnider_dataset(df: pd.DataFrame):
    params_list = []
    for _, row in df.iterrows():
        params = schnider_model(
            age=row["AgeCategory"],
            weight=row["WeightInKilograms"],
            height=row["HeightInMeters"],
            sex=row["Sex"]
        )
        params_list.append(params)

    params_df = pd.DataFrame(params_list)
    return pd.concat([df.reset_index(drop=True), params_df], axis=1)

# 2. Evaluator for DP Agent (100 states, 2 features: BIS error and deltaBIS)
class DPEvaluator:
    def __init__(self, policy, actions):
        self.policy = policy
        self.actions = actions
        self.target = 50.0
        self.bis0 = 95.0
        self.bis_max = 75.0
        self.ec50 = 3.5
        self.hill = 2.5
        self.bins_per_feat = 10

    def _get_state_idx(self, bis_error, deltabis):
        """Match feature extraction from DP training exactly"""
        # Create fuzzy features from bis_error and deltabis
        error_scaled = np.clip(bis_error / 20.0, -1, 1)
        mu_error_neg = max(0, -error_scaled)
        mu_error_pos = max(0, error_scaled)

        delta_scaled = np.clip(deltabis / 20.0, -1, 1)
        mu_delta_neg = max(0, -delta_scaled)
        mu_delta_pos = max(0, delta_scaled)

        features = np.array([mu_error_neg, mu_error_pos, mu_delta_neg, mu_delta_pos])

        # Discretize to state index
        error_feat = (features[1] - features[0]) * (self.bins_per_feat - 1)
        delta_feat = (features[3] - features[2]) * (self.bins_per_feat - 1)

        error_bin = int(np.clip(error_feat, 0, self.bins_per_feat - 1))
        delta_bin = int(np.clip(delta_feat, 0, self.bins_per_feat - 1))

        idx = error_bin * self.bins_per_feat + delta_bin
        idx = min(idx, len(self.policy) - 1)
        return idx

    def simulate(self, patient_row, duration_min=120):
        dt = 5/60
        steps = int(duration_min / dt)

        A = np.asarray(patient_row['A'], dtype=float)
        B = np.asarray(patient_row['B'], dtype=float)

        x = np.zeros((4, 1), dtype=float)

        pe_log = []
        bis_log = []
        bis_prev_prev = self.bis0
        bis_prev = self.bis0

        for t in range(steps):
            ce = max(0.0, x[3, 0])

            ce_h = ce ** self.hill
            ec50_h = self.ec50 ** self.hill
            bis_ideal = self.bis0 - self.bis_max * (ce_h / (ce_h + ec50_h))

            if np.isnan(bis_ideal):
                bis_ideal = self.bis0

            bis_ideal = np.clip(bis_ideal, 0, 100)
            measured_bis = bis_ideal + np.random.normal(0, 3)
            measured_bis = np.clip(measured_bis, 0, 100)

            # Compute state from BIS error and deltaBIS
            bis_error = measured_bis - self.target
            deltabis = measured_bis - bis_prev_prev if t > 0 else 0.0

            s_idx = self._get_state_idx(bis_error, deltabis)

            a_idx = int(self.policy[s_idx])
            u = float(self.actions[a_idx])

            x_dot = A @ x + B * u
            x = x + x_dot * dt

            bis_prev_prev = bis_prev
            bis_prev = measured_bis

            error = measured_bis - self.target
            pe_log.append(100.0 * error / self.target if self.target != 0 else 0.0)
            bis_log.append(float(measured_bis))

        return bis_log, pe_log

    def evaluate(self, df):
        summary = []
        for idx, row in df.iterrows():
            try:
                bis, pe = self.simulate(row)
                mdpe = np.median(pe)
                mdape = np.median(np.abs(pe))
                wobble = np.median(np.abs(pe - mdpe))
                controlled = (np.abs(np.array(bis) - self.target) <= 5).mean() * 100
                summary.append({
                    'PatientID': row['PatientID'],
                    'MDPE': mdpe, 'MDAPE': mdape,
                    'Wobble': wobble, 'Controlled (%)': controlled
                })
            except Exception as e:
                print(f"Warning: Failed to evaluate patient {row['PatientID']}: {e}")
                continue
        return pd.DataFrame(summary)

# 3. Execution
EVAL_SAMPLE_SIZE = 50

try:
    print("Loading Population Data...")
    raw_data = load_data("./data/Patients Data.csv")
    cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
    df_clean = preprocess_data(raw_data[cols])

    if EVAL_SAMPLE_SIZE and len(df_clean) > EVAL_SAMPLE_SIZE:
        print(f"Sampling {EVAL_SAMPLE_SIZE} patients from {len(df_clean)} total.")
        df_clean = df_clean.sample(n=EVAL_SAMPLE_SIZE, random_state=42)

    df_sim = generate_schnider_dataset(df_clean)

    print("Loading DP Agent...")
    dp_data = np.load(DP_PATH)
    loaded_policy = dp_data["policy"]
    loaded_actions = dp_data["actions"]

    print("Evaluating on Population...")
    evaluator = DPEvaluator(loaded_policy, loaded_actions)
    results_df = evaluator.evaluate(df_sim)

    print("\n--- Evaluation Results Summary ---")
    print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

    best_pt = results_df.loc[results_df['MDAPE'].idxmin()]
    print(f"\nBest Patient: {best_pt.to_dict()}")

except Exception as e:
    print(f"Evaluation failed: {e}")


# LOAD AND EVALUATE SAVED DP AGENT

In [ ]:
print("=" * 60)
print("LOADING SAVED DP AGENT (BIS + deltaBIS with 2 membership functions)")
print("=" * 60)

if not DP_PATH.exists():
    print(f"ERROR: Agent file not found at {DP_PATH}")
    print("Please run the DP training cell first.")
else:
    try:
        print(f"\nLoading agent from: {DP_PATH}")
        agent_data = np.load(DP_PATH)

        print(f"  - Loaded V (value function): shape {agent_data['V'].shape}")
        print(f"  - Loaded policy: shape {agent_data['policy'].shape}")
        print(f"  - Loaded P (transitions): shape {agent_data['P'].shape}")
        print(f"  - Loaded R (rewards): shape {agent_data['R'].shape}")
        print(f"  - Actions available: {agent_data['actions']}")
        print(f"  - Gamma (discount): {agent_data['gamma']}")

        saved_policy = agent_data["policy"]
        saved_actions = agent_data["actions"]

        evaluator = DPEvaluator(saved_policy, saved_actions)

        print("\n" + "=" * 60)
        print("EVALUATING ON POPULATION SAMPLE")
        print("=" * 60)

        raw_data = load_data("./data/Patients Data.csv")
        cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
        df_clean = preprocess_data(raw_data[cols])

        sample_size = 100
        if len(df_clean) > sample_size:
            print(f"\nSampling {sample_size} patients from {len(df_clean)} total...")
            df_clean = df_clean.sample(n=sample_size, random_state=42)

        print(f"Generating Schnider parameters for {len(df_clean)} patients...")
        df_sim = generate_schnider_dataset(df_clean)

        print(f"\nRunning evaluation simulation (120 min per patient)...")
        results_df = evaluator.evaluate(df_sim)

        if len(results_df) > 0:
            print("\n" + "=" * 60)
            print("EVALUATION RESULTS")
            print("=" * 60)
            print(f"\nEvaluated {len(results_df)} patients successfully\n")
            print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

            best_idx = results_df['MDAPE'].idxmin()
            worst_idx = results_df['MDAPE'].idxmax()

            print(f"\n--- Best Controlled Patient (ID: {results_df.loc[best_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[best_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[best_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[best_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[best_idx, 'Controlled (%)']:.2f}%")

            print(f"\n--- Worst Controlled Patient (ID: {results_df.loc[worst_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[worst_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[worst_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[worst_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[worst_idx, 'Controlled (%)']:.2f}%")
        else:
            print("No patients were successfully evaluated.")

    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        traceback.print_exc()
